# Lab: Sentiment Analysis  
#  *******Data-Centric vs Model-Centric approaches




This lab gives an introduction to sentiment analysis approaches.

In this lab, we'll build a classifier for product reviews (restricted to the magazine category), like:

> Excellent! I look forward to every issue. I had no idea just how much I didn't know.  The letters from the subscribers are educational, too.

Label: ⭐️⭐️⭐️⭐️⭐️ (good)

> My son waited and waited, it took the 6 weeks to get delivered that they said it would but when it got here he was so dissapointed, it only took him a few minutes to read it.

Label: ⭐️ (bad)

We'll work with a dataset that has some issues, and we'll see how we can squeeze only so much performance out of the model by being clever about model choice, searching for better hyperparameters, etc. Then, we'll take a look at the data (as any good data scientist should), develop an understanding of the issues, and use simple approaches to improve the data. Finally, we'll see how improving the data can improve results.

## Installing software

For this lab, you'll need to install [scikit-learn](https://scikit-learn.org/) and [pandas](https://pandas.pydata.org/). If you don't have them installed already, you can install them by running the following cell:

In [1]:
!pip install scikit-learn pandas

# Loading the data

First, let's load the train/test sets and take a look at the data.

In [2]:
import pandas as pd

In [4]:
train = pd.read_csv('reviews_train.csv')
test = pd.read_csv('reviews_test.csv')

test.sample(5)

,review,label
415,This is the second year that I purchased this ...,good
636,"This magazine has gone to,,,you know the emoji...",bad
681,The magazine is so thin -- not much articles t...,bad
36,"Excellent magazine, wonderful and informative.",good
328,My mom loves w so I ordered this for her.,good


# Training a baseline model

There are many approaches for training a sequence classification model for text data. In this lab, we're giving you code that mirrors what you find if you look up [how to train a text classifier](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html), where we'll train an SVM on [tf-idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) features (numeric representations of each text field based on word occurrences).

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

In [6]:
sgd_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

In [7]:
_ = sgd_clf.fit(train['review'], train['label'])

## Evaluating model accuracy

In [8]:
from sklearn import metrics

In [9]:
def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

In [10]:
evaluate(sgd_clf)

Accuracy: 77.1%


## Trying another model

76% accuracy is not great for this binary classification problem. Can you do better with a different model, or by tuning hyperparameters for the SVM trained with SGD?

# Exercise 1

Can you train a more accurate model on the dataset (without changing the dataset)? You might find this [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) handy, as well as the [documentation for supervised learning in scikit-learn](https://scikit-learn.org/stable/supervised_learning.html).

One idea for a model you could try is a [naive Bayes classifier](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

You could also try experimenting with different values of the model hyperparameters, perhaps tuning them via a [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Or you can even try training multiple different models and [ensembling their predictions](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), a strategy often used to win prediction competitions like Kaggle.

**Advanced:** If you want to be more ambitious, you could try an even fancier model, like training a Transformer neural network. If you go with that, you'll want to fine-tune a pre-trained model. This [guide from HuggingFace](https://huggingface.co/docs/transformers/training) may be helpful.

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [12]:
# YOUR CODE HERE
best_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', LinearSVC())
])
# evaluate your model and see if it does better
# than the ones we provided

In [18]:
parameters = {
    'vect__ngram_range': [(1,1), (1,2), (1,3)],
    'vect__max_df': [0.85, 0.9, 0.95],
    'vect__min_df': [1, 2, 3],
    'vect__stop_words': [None, 'english'],
    'tfidf__sublinear_tf': [True, False],
    'clf__C': [1,5]
}

In [19]:
grid_search = GridSearchCV(
    best_clf,
    parameters,
    cv=3,
    n_jobs=-1,
    scoring='accuracy'
)

In [20]:
_ = grid_search.fit(train['review'], train['label'])

In [21]:
predicted = grid_search.predict(test['review'])

In [22]:
print("Test Accuracy:", accuracy_score(test['label'], predicted))

Test Accuracy: 0.801


In [25]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

pipeline = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', LinearSVC())
])

parameters = [
    {
        'clf': [LinearSVC()],
        'clf__C': [0.1, 1, 10]
    },
    {
        'clf': [LogisticRegression(max_iter=5000)],
        'clf__C': [0.1, 1, 10]
    }
]

grid = GridSearchCV(pipeline, parameters, cv=5, scoring='accuracy', n_jobs=-1)
_ = grid.fit(train['review'], train['label'])
predicted = grid.predict(test['review'])

print("Best classifier:", grid.best_estimator_)
print("Test Accuracy:", accuracy_score(test['label'], predicted))

Best classifier: Pipeline(steps=[('vect', CountVectorizer()), ('tfidf', TfidfTransformer()),
                ('clf', LinearSVC(C=0.1))])
Test Accuracy: 0.788


## Taking a closer look at the training data

Let's actually take a look at some of the training data:

In [26]:
train.head()

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


Zooming in on one particular data point:

In [27]:
print(train.iloc[0].to_dict())

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}


This data point is labeled "good", but it's clearly a negative review. Also, it looks like there's some funny HTML stuff at the end.

# Exercise 2

Take a look at some more examples in the dataset. Do you notice any patterns with bad data points?

In [28]:
# YOUR CODE HERE
sample_rows = train.sample(10, random_state=42)

for idx, row in sample_rows.iterrows():
    print(f"Index: {idx}")
    print("Label:", row['label'])
    print("Review:", row['review'])
    print("-" * 100)

Index: 239
Label: bad
Review: Signed up at a low rate but the auto renewal rate was a ripoff too high. Decided to cancel it.
----------------------------------------------------------------------------------------------------
Index: 2272
Label: good
Review: </span>I can't figure out how to download it to my Kindle.</li>
----------------------------------------------------------------------------------------------------
Index: 6173
Label: bad
Review: Owned by Fox and is now filled with a ton of ads.
----------------------------------------------------------------------------------------------------
Index: 2575
Label: good
Review: It didn't meet my expectations.<div id=nav>
----------------------------------------------------------------------------------------------------
Index: 5510
Label: bad
Review: misleading - says full access - u don't get the web lookup site access .. only the magazine in paper and digital form
---------------------------------------------------------------------

In [29]:
#I noticed some bad data patterns in the dataset. Some reviews seem to have incorrect labels, where the text sentiment does not match the assigned label. I also noticed noisy text such as HTML tags and formatting artifacts inside some reviews

## Issues in the data

It looks like there's some funny HTML tags in our dataset, and those datapoints have nonsense labels. Maybe this dataset was collected by scraping the internet, and the HTML wasn't quite parsed correctly in all cases.

# Exercise 3

To address this, a simple approach we might try is to throw out the bad data points, and train our model on only the "clean" data.

Come up with a simple heuristic to identify data points containing HTML, and filter out the bad data points to create a cleaned training set.

In [39]:
import re
def is_bad_data(review: str) -> bool:
    if not isinstance(review, str):
        return True
    return bool(re.search(r"<[^>]+>", review))

## Creating the cleaned training set

In [40]:
train_clean = train[~train['review'].map(is_bad_data)]

## Evaluating a model trained on the clean training set

In [41]:
from sklearn import clone

In [42]:
sgd_clf_clean = clone(sgd_clf)

In [43]:
_ = sgd_clf_clean.fit(train_clean['review'], train_clean['label'])

This model should do significantly better:

In [44]:
evaluate(sgd_clf_clean)

Accuracy: 97.1%
